In [10]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import re
import time

In [11]:
def fetch_kb_html(date_str: str, session: requests.Session) -> str:
    URL = "https://obank.kbstar.com/quics?chgCompId=b102292&baseCompId=b102292&page=C101408&cc=b102292:b102292"

    HEADERS = {
        "User-Agent": "Mozilla/5.0",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": "https://obank.kbstar.com",
        "Referer": "https://obank.kbstar.com/quics?page=C101408",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    }

    payload = {
        "STEP": "0",
        "조회년월일": date_str,
        "strFocusBtn": "",
        "tabNumber": "",
        "MMDA고객구분": "",
        "se_inqueryYYYY": date_str[:4],
        "se_inqueryMM": date_str[4:6],
        "se_inqueryDD": date_str[6:8],
        "MMDA통화코드": "USD",
    }

    r = session.post(URL, data=payload, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

In [12]:
session = requests.Session()

html = fetch_kb_html("20260325", session)

print(html[:500])

<!-- b102292 START --><div id="b102292">

<script type="text/javascript">
//<![CDATA[
    $().ready(function(){
        //input text 하나있을경우 엔터키 적용제거
        $('#CP form').submit(function(){
            return false;
        });

        //모바일기기시 인쇄 저장버튼 제거
        if(caq.util.isPadFlag() || caq.util.isAndroidFlag() || caq.util.isIOSFlag()){
            $('#CP input[type=button][value=저장]').hide();
            $('#CP input[type=button][value=인쇄]').hide();
            $('#CP a:contain


In [13]:
BANK_NAME = "KB Kookmin Bank"
BANK_CODE = "KB"

URL = "https://obank.kbstar.com/quics?chgCompId=b102292&baseCompId=b102292&page=C101408&cc=b102292:b102292"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://obank.kbstar.com",
    "Referer": "https://obank.kbstar.com/quics?page=C101408",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
}

SAVE_DIR = Path("./outputs_kb_fx")
SAVE_DIR.mkdir(exist_ok=True)

In [14]:
def generate_quarter_dates(start=2004, end=2019):
    dates = []
    for y in range(start, end + 1):
        dates.extend([
            f"{y}0331",
            f"{y}0630",
            f"{y}0930",
            f"{y}1231",
        ])
    return dates

target_dates = generate_quarter_dates(2004, 2019)
target_dates[:8], target_dates[-4:]

(['20040331',
  '20040630',
  '20040930',
  '20041231',
  '20050331',
  '20050630',
  '20050930',
  '20051231'],
 ['20190331', '20190630', '20190930', '20191231'])

In [15]:
def generate_quarter_dates(start=2004, end=2019):
    dates = []
    for y in range(start, end + 1):
        dates.extend([
            f"{y}0331",
            f"{y}0630",
            f"{y}0930",
            f"{y}1231",
        ])
    return dates

target_dates = generate_quarter_dates(2004, 2019)
target_dates[:8], target_dates[-4:]

(['20040331',
  '20040630',
  '20040930',
  '20041231',
  '20050331',
  '20050630',
  '20050930',
  '20051231'],
 ['20190331', '20190630', '20190930', '20191231'])

In [16]:
def clean_text(x):
    if x is None:
        return None
    x = str(x).strip()
    return x if x != "" else None

def clean_rate(x):
    x = clean_text(x)
    if x in [None, "-", ""]:
        return None
    x = x.replace(",", "")
    try:
        return float(x)
    except:
        return None

def clean_currency(x):
    x = clean_text(x)
    if x is None:
        return None
    m = re.match(r"([A-Z]{3})", x)
    return m.group(1) if m else x

def extract_observed_date(html: str, fallback_date: str) -> str:
    m = re.search(r"조회기준일\s*:\s*(\d{4})\.(\d{2})\.(\d{2})", html)
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    return f"{fallback_date[:4]}-{fallback_date[4:6]}-{fallback_date[6:8]}"

In [17]:
def parse_simple_rate_table(table_tag, target_date, observed_date, product_group, residency):
    rows = []

    tr_list = table_tag.find_all("tr")
    if not tr_list:
        return pd.DataFrame()

    header_cells = tr_list[0].find_all(["th", "td"])
    headers = [c.get_text(" ", strip=True) for c in header_cells]

    for tr in tr_list[1:]:
        cells = tr.find_all(["td", "th"])
        values = [c.get_text(" ", strip=True) for c in cells]

        if len(values) < 2:
            continue

        currency = clean_currency(values[0])
        if currency is None:
            continue

        n = min(len(headers), len(values))

        for i in range(1, n):
            maturity = headers[i]
            rate = clean_rate(values[i])

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": f"{target_date[:4]}-{target_date[4:6]}-{target_date[6:8]}",
                "observed_date": observed_date,
                "currency": currency,
                "residency": residency,
                "maturity": maturity,
                "rate": rate,
                "product_group": product_group,
                "unit": "annual % (pretax)",
            })

    return pd.DataFrame(rows)

In [18]:
def parse_matrix_table(table_tag, target_date, observed_date, product_group):
    rows = []

    tr_list = table_tag.find_all("tr")
    if len(tr_list) < 3:
        return pd.DataFrame()

    # 첫 두 행 헤더 읽기
    header_rows = tr_list[:2]
    body_rows = tr_list[2:]

    h1 = [c.get_text(" ", strip=True) for c in header_rows[0].find_all(["th", "td"])]
    h2 = [c.get_text(" ", strip=True) for c in header_rows[1].find_all(["th", "td"])]

    # 첫 열은 통화
    # 나머지는 조합헤더로 생성
    combined_headers = ["통화"]

    if len(h1) >= 2 and len(h2) >= 1:
        # 보통예금 유형: 거주자/비거주자 + 보통예금/통지예금
        if "거주자" in " ".join(h1) or "비거주자" in " ".join(h1):
            # h1 첫칸은 통화, 이후 그룹
            groups = []
            for c in h1[1:]:
                if c != "":
                    groups.append(c)

            # h2 길이에 맞춰 펴기
            # 보통 2개씩 거주자/비거주자 반복
            if len(groups) == 2 and len(h2) == 4:
                combined_headers = [
                    "통화",
                    f"{groups[0]}|{h2[0]}",
                    f"{groups[0]}|{h2[1]}",
                    f"{groups[1]}|{h2[2]}",
                    f"{groups[1]}|{h2[3]}",
                ]
            else:
                combined_headers = ["통화"] + h2
        else:
            combined_headers = ["통화"] + h2

    for tr in body_rows:
        cells = tr.find_all(["td", "th"])
        values = [c.get_text(" ", strip=True) for c in cells]

        if len(values) < 2:
            continue

        currency = clean_currency(values[0])
        if currency is None:
            continue

        n = min(len(combined_headers), len(values))
        for i in range(1, n):
            col = combined_headers[i]
            rate = clean_rate(values[i])

            residency = None
            maturity = col

            if "|" in col:
                residency, maturity = col.split("|", 1)
                if residency == "거주자":
                    residency = "resident"
                elif residency == "비거주자":
                    residency = "nonresident"

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": f"{target_date[:4]}-{target_date[4:6]}-{target_date[6:8]}",
                "observed_date": observed_date,
                "currency": currency,
                "residency": residency,
                "maturity": maturity,
                "rate": rate,
                "product_group": product_group,
                "unit": "annual % (pretax)",
            })

    return pd.DataFrame(rows)

In [19]:
def parse_kb_html(html: str, target_date: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "lxml")
    observed_date = extract_observed_date(html, target_date)

    all_parts = []

    # 1) 정기예금(거주자) - 확장표 우선
    t = soup.find("div", {"id": "targetTable1"})
    if t:
        table = t.find("table")
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "정기예금", "resident"
            ))
    else:
        table = soup.find("table", {"id": "viewType_1_0"})
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "정기예금", "resident"
            ))

    # 2) 정기예금(비거주자) - 확장표 우선
    t = soup.find("div", {"id": "targetTable2"})
    if t:
        table = t.find("table")
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "정기예금", "nonresident"
            ))
    else:
        table = soup.find("table", {"id": "viewType_2_0"})
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "정기예금", "nonresident"
            ))

    # 3) 보통예금
    table = soup.find("table", {"id": "viewType_3_0"})
    if table:
        all_parts.append(parse_matrix_table(
            table, target_date, observed_date, "보통예금"
        ))

    # 4) KB수출입기업우대외화통장
    table = soup.find("table", {"id": "viewType_4_0"})
    if table:
        all_parts.append(parse_matrix_table(
            table, target_date, observed_date, "KB수출입기업우대외화통장"
        ))

    # 5) KB국민UP외화정기예금 - 확장표 우선
    t = soup.find("div", {"id": "targetTable3"})
    if t:
        table = t.find("table")
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "KB국민UP외화정기예금", "resident"
            ))
    else:
        table = soup.find("table", {"id": "viewType_5_0"})
        if table:
            all_parts.append(parse_simple_rate_table(
                table, target_date, observed_date, "KB국민UP외화정기예금", "resident"
            ))

    # 6) MMDA (개인)
    table = soup.find("table", {"id": "viewType_6_0"})
    if table:
        all_parts.append(parse_matrix_table(
            table, target_date, observed_date, "KB외화MMDA(개인)"
        ))

    # 7) 외화적금
    table = soup.find("table", {"id": "viewType_7_0"})
    if table:
        all_parts.append(parse_simple_rate_table(
            table, target_date, observed_date, "외화적금", "resident"
        ))

    if not all_parts:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date", "observed_date", "currency",
            "residency", "maturity", "rate", "product_group", "unit"
        ])

    out = pd.concat(all_parts, ignore_index=True)
    return out

In [20]:
session = requests.Session()

test_date = "20260325"
html = fetch_kb_html(test_date, session)
df_test = parse_kb_html(html, test_date)

print("shape:", df_test.shape)
print(df_test.head(30))
print(df_test["product_group"].value_counts(dropna=False))
print(sorted(df_test["currency"].dropna().unique().tolist())[:20])

shape: (399, 10)
               bank bank_code target_date observed_date currency residency  \
0   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
1   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
2   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
3   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
4   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
5   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
6   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
7   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
8   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
9   KB Kookmin Bank        KB  2026-03-25    2026-03-25      USD  resident   
10  KB Kookmin Bank        KB  2026-03-25    2026-03-25      JPY  resident   
11  KB Kookmin Bank        KB  2026-03-25    20

In [21]:
session = requests.Session()

all_parts = []
fail_log = []

for i, d in enumerate(target_dates, start=1):
    try:
        html = fetch_kb_html(d, session)

        if "고객님 죄송합니다" in html and "errorDiv" in html:
            # 에러 div가 있어도 실제 표는 같이 있을 수 있으므로 먼저 파싱 시도
            pass

        df = parse_kb_html(html, d)

        if df.empty:
            fail_log.append({"date": d, "reason": "empty"})
        else:
            all_parts.append(df)
            print(f"{d} OK ({len(df)} rows)")

        time.sleep(0.4)

    except Exception as e:
        fail_log.append({"date": d, "reason": str(e)})

    if i % 10 == 0:
        print(f"{i}/{len(target_dates)} done")

kb_all = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame(columns=[
    "bank", "bank_code", "target_date", "observed_date", "currency",
    "residency", "maturity", "rate", "product_group", "unit"
])

kb_fail = pd.DataFrame(fail_log)

print("FINAL SHAPE:", kb_all.shape)
print("FAIL COUNT:", len(kb_fail))

10/64 done
20/64 done
20110331 OK (256 rows)
20110630 OK (256 rows)
30/64 done
20110930 OK (256 rows)
20121231 OK (256 rows)
20130930 OK (256 rows)
20131231 OK (256 rows)
40/64 done
20140331 OK (256 rows)
20140630 OK (256 rows)
20140930 OK (256 rows)
20141231 OK (256 rows)
20150331 OK (256 rows)
20150630 OK (256 rows)
20150930 OK (256 rows)
20151231 OK (256 rows)
20160331 OK (256 rows)
20160630 OK (256 rows)
50/64 done
20160930 OK (256 rows)
20170331 OK (256 rows)
20170630 OK (256 rows)
20181231 OK (256 rows)
60/64 done
20190930 OK (256 rows)
20191231 OK (256 rows)
FINAL SHAPE: (5632, 10)
FAIL COUNT: 42


In [22]:
if not kb_all.empty:
    kb_all["target_date"] = pd.to_datetime(kb_all["target_date"])
    kb_all["observed_date"] = pd.to_datetime(kb_all["observed_date"], errors="coerce")
    kb_all["rate"] = pd.to_numeric(kb_all["rate"], errors="coerce")

    # 통화 코드 정리
    kb_all["currency"] = kb_all["currency"].astype(str).str.extract(r"([A-Z]{3})")

    kb_all = kb_all.sort_values(
        ["target_date", "product_group", "residency", "currency", "maturity"]
    ).reset_index(drop=True)

print(kb_all.head(30))

               bank bank_code target_date observed_date currency residency  \
0   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
1   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
2   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
3   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
4   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
5   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
6   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
7   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
8   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
9   KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
10  KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR  resident   
11  KB Kookmin Bank        KB  2011-03-31    2011-03-31      EUR

In [23]:
output_path = SAVE_DIR / "kb_fx_2004_2019.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    kb_all.to_excel(writer, sheet_name="raw_all", index=False)
    kb_fail.to_excel(writer, sheet_name="fail_log", index=False)

    if not kb_all.empty:
        summary = (
            kb_all.groupby(["product_group", "residency", "currency"])["rate"]
            .agg(["count", "min", "max"])
            .reset_index()
        )
        summary.to_excel(writer, sheet_name="summary", index=False)

print("saved:", output_path)

saved: outputs_kb_fx\kb_fx_2004_2019.xlsx
